<div style="box-sizing:border-box; min-height:12.65in; border:5px solid black; border-radius:6px; padding:48px; display:flex; flex-direction:column; justify-content:center; text-align:center;">
<h1>Embeddings and Vector Management in Practice</h1>
<h3>Author: Tim Hollis</h3>
<h3>Bellevue University</h3>
<h3>Course: DSC670 - Advanced Uses of Generative AI</h3>
<h3>Professor Neugebauer</h3>
<h3>Date: August 1, 2026</h3>
</div>

<br><br>

<div style="font-family: 'Calibri', serif; font-size: 12pt; line-height: 2;">

<p style="text-align: center;"><b>Embeddings and Vector Management in Practice</b></p>

<p style="text-indent: 0.5in;">An embedding is a list of numbers that stands in for the meaning of a piece of content. The text reading treats this as a preprocessing step, but in a production generative AI system, the embedding is the retrieval system. When a user asks a question, the application does not search documents for matching words; it converts the question into a vector and looks for the stored vectors that point in a similar direction. Everything the model eventually says about private data depends on whether that geometric lookup returned the right passages. Understanding embeddings therefore means understanding two distinct problems: how a model produces a vector, and how an organization stores, indexes, and maintains millions of them over time.</p>

<p style="text-align: left;"><b>How Embedding Models Turn Meaning Into Geometry</b></p>

<p><b>The Vector as a Coordinate</b></p>

<p style="text-indent: 0.5in;">A modern embedding model maps content into a high-dimensional space where proximity means similarity. Google's gemini-embedding-2 returns a 3,072-dimensional vector by default, and similarity between two vectors is normally measured with cosine similarity, which compares direction rather than magnitude and returns a value between -1 and 1 (Google, 2026). Direction matters more than length here because two passages can differ in verbosity while expressing the same idea. This is also why normalization is an operational detail worth knowing: Google's older gemini-embedding-001 requires manual normalization whenever a developer requests fewer than 3,072 dimensions, while gemini-embedding-2 renormalizes truncated vectors automatically (Google, 2026). A team that upgraded without noticing that difference would silently degrade its own similarity scores.</p>

<p><b>Task-Specific Embeddings</b></p>

<p style="text-indent: 0.5in;">A practical wrinkle the reading does not emphasize is that embedding models are tuned for the job being asked of them. With gemini-embedding-001, a developer passes a task_type parameter such as RETRIEVAL_DOCUMENT, RETRIEVAL_QUERY, or CLASSIFICATION, and the returned vector is optimized for that relationship (Google, 2026). Retrieval is asymmetric: a short question and the long passage that answers it are not the same kind of object, so they are embedded differently. Gemini Embedding 2 removed the parameter and instead expects the task to be stated as a prefix in the input itself, formatting a query as a search or question-answering task and the stored content with a title and text structure (Google, 2026). The governing rule in both versions is consistency. If a corpus is indexed one way and queried another, retrieval quality drops for reasons that never appear in an error log.</p>

<br><br><br><br><br><br><br>

<p style="text-align: left;"><b>Vector Management as an Engineering Discipline</b></p>

<p><b>Choosing a Dimension</b></p>

<p style="text-indent: 0.5in;">Dimension count is a direct cost decision. Storing a million passages at 3,072 dimensions consumes four times the memory of storing them at 768, and every similarity comparison touches four times as many numbers. Both Google and OpenAI address this with Matryoshka Representation Learning, a training technique that packs the most important information toward the front of the vector so that a truncated prefix remains a usable embedding (Kusupati et al., 2022). The measured tradeoff is surprisingly mild. Google's published MTEB scores for gemini-embedding-001 show 68.17 at 1,536 dimensions and 67.99 at 768, essentially matching the full 3,072-dimension score of 68.17 (Google, 2026). OpenAI implemented the same idea as a dimensions parameter on text-embedding-3-small and text-embedding-3-large, and reported that a 3-large embedding shortened to 256 dimensions still outperformed the older text-embedding-ada-002 at 1,536 (OpenAI, 2024). That flexibility solves a real constraint: a team using a vector store capped at 1,024 dimensions can still use the stronger model and trim the output to fit.</p>

<p><b>Indexing for Approximate Search</b></p>

<p style="text-indent: 0.5in;">Comparing a query vector against every stored vector is exact but does not scale. Production systems instead use approximate nearest neighbor indexes, most commonly Hierarchical Navigable Small World graphs, which build a layered proximity graph and navigate from coarse long-range links down to fine local ones (Malkov & Yashunin, 2020). The index trades a small amount of recall for a large gain in speed, and its tuning parameters are where that trade is set. This is the part of vector management that most resembles traditional database administration: an index that is cheap to build and fast to query will quietly miss relevant documents, and the only way to know is to measure recall against a known answer set.</p>

<p style="text-align: left;"><b>The Migration Problem</b></p>

<p style="text-indent: 0.5in;">The most consequential practical fact about vector management is that embeddings are not portable across models. Google states plainly that the embedding spaces of gemini-embedding-001 and gemini-embedding-2 are incompatible, that vectors from one cannot be compared against vectors from the other, and that upgrading requires re-embedding the entire corpus (Google, 2026). For an organization with millions of stored documents, that is a billing event and a migration project, not a configuration change. The same logic applies to dimension choice, since changing the target dimension after ingestion means re-embedding as well. A vector store is therefore a durable architectural commitment rather than a cache, and the decision about which model and dimension to use should be made before the first document is indexed.</p>

<p style="text-align: left;"><b>Conclusion</b></p>

<p style="text-indent: 0.5in;">Embeddings look like a single API call, and vector management looks like a storage question. In practice, they are one coupled system whose parameters- the model, the task type, the dimension count, and the index- all determine what a generative AI application can find and therefore what it can say. The vendor documentation reads less like machine learning theory and more like operations guidance, which is a useful signal about where the real work lives.</p>

<br><br>

<p style="text-align: center;"><b>References</b></p>

<p style="text-indent: -0.5in; padding-left: 0.5in;">Google. (2026). <i>Embeddings</i>. Gemini API documentation. https://ai.google.dev/gemini-api/docs/embeddings</p>

<p style="text-indent: -0.5in; padding-left: 0.5in;">Kusupati, A., Bhatt, G., Rege, A., Wallingford, M., Sinha, A., Ramanujan, V., Howard-Snyder, W., Chen, K., Kakade, S., Jain, P., &amp; Farhadi, A. (2022). <i>Matryoshka representation learning</i> (arXiv:2205.13147). arXiv. https://arxiv.org/abs/2205.13147</p>

<p style="text-indent: -0.5in; padding-left: 0.5in;">Malkov, Y. A., &amp; Yashunin, D. A. (2020). Efficient and robust approximate nearest neighbor search using hierarchical navigable small world graphs. <i>IEEE Transactions on Pattern Analysis and Machine Intelligence, 42</i>(4), 824–836. https://doi.org/10.1109/TPAMI.2018.2889473</p>

<p style="text-indent: -0.5in; padding-left: 0.5in;">OpenAI. (2024, January 25). <i>New embedding models and API updates</i>. https://openai.com/index/new-embedding-models-and-api-updates/</p>

</div>

### **Reflection for Week 8**

#### General Reflection

This week was the point where the course's conceptual pieces connected to the engineering reality underneath them. The reading covered several orchestration topics at a shallow depth, but digging into embeddings and vector management revealed how much of a generative AI system's reliability depends on decisions that never appear in the model's output. As I wrote in the paper, "an embedding is a list of numbers that stands in for the meaning of a piece of content," but in practice it becomes the entire retrieval layer, and the model's answer is only as good as the geometric lookup that precedes it.

Building the model reinforced how much sits below the surface. The notebook made every hidden step explicit: constructing the training data, masking the loss, handling truncation, and verifying that the fine-tune was truly full rather than adapter-based. Seeing the audit printout confirm "Every parameter is trainable and no adapter is present" made the requirement concrete instead of asserted. The migration problem became real in the same way once I saw that embedding spaces are incompatible across model versions. Upgrading is not a configuration change, it is a re-embedding event, and that has operational consequences.

#### Straightforward Aspects

- Writing the embeddings paper was smooth because the topic lends itself to practical examples. The differences between gemini-embedding-001 and gemini-embedding-2, especially around normalization and task formatting, made it easy to explain why consistency matters.
- The vector management section came together quickly. The Matryoshka Representation Learning examples from Google and OpenAI provided clear evidence that dimension reduction is not just theoretical but operationally useful.
- Mirroring the Chapter 7 metrics was easier than expected once the loss masking was correct. Pairing each validation pass with a matching pass over held-in data produced all four series under their original names without much additional work.
- Output format turned out to be the solvable half of the problem, exactly as the Milestone 2 experiments predicted. The fine-tuned model produced a parseable verdict on all 441 held-out postings, while the untuned base model produced zero.

#### Challenging Aspects

- Constructing the training data was the most iterated work of the week, not the least. The first version cited company branding that the model could not observe in its own input, which taught it to assert an unverifiable fact. The second version collapsed 16,624 postings into 61 unique response strings, so the targets were boilerplate rather than explanations. Confidence also turned out to be determined by the verdict rather than the evidence, with most legitimate postings at HIGH and most fraudulent ones at MEDIUM. Each of those took a rewrite to fix.
- Deciding what to do with the 30% of frauds that had no expressible evidence required a judgment call rather than a technique. Training on targets that assert fraud without a checkable reason would teach unexplainable behavior, so those postings were excluded from training and evaluated separately as a labeled group.
- Oversampling polished frauds and unusual legitimate postings added real complexity to the split logic. My first version claimed to oversample polished frauds but only reproduced their natural rate, since drawing uniformly from the fraud pool cannot raise the share of a subset within it. Fixing that required explicit repetition in the training split alone.
- The most uncomfortable finding was in my own model. Across 441 responses it quoted 82 phrases, and 11 of them do not appear in the postings they describe. Every fabricated quote came from the vocabulary I used to construct the training targets, with two terms accounting for most of them. Constructed data teaches the wording of the construction and not only its logic, which is a limitation I would not have predicted before seeing it.